### Read in NYISO Parquests

In [ ]:
# Read and aggregate master parquet file
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import time

# Add Build directory to path for path_utils
sys.path.insert(0, str(Path.cwd() / "Build"))
from path_utils import get_project_root, MASTER_PARQUET

# Get project root and master path
project_root = get_project_root()
MASTER_PATH = MASTER_PARQUET

print(f"Project root: {project_root}")
print(f"Loading data from: {MASTER_PATH}")

# Load only necessary columns to reduce memory usage
print("Reading parquet file (this may take 10-20 seconds)...")
start = time.time()
df_all = pd.read_parquet(MASTER_PATH, columns=['datetime', 'Load'])
print(f"✓ Loaded {len(df_all):,} rows in {time.time()-start:.1f}s")

# Ensure datetime column is properly formatted
print("Converting datetime...")
df_all['datetime'] = pd.to_datetime(df_all['datetime'], utc=True)

# Filter for 2023 data only (to match original visualization)
print("Filtering to 2023...")
df_all = df_all[df_all['datetime'].dt.year == 2023].copy()
print(f"✓ Filtered to 2023: {len(df_all):,} rows")

# Create time-based aggregates
print("Creating aggregates...")
df_all = df_all.set_index('datetime')

# Aggregate to total load per timestamp (sum across all zones/names)
df_grouped = df_all.groupby(df_all.index)['Load'].sum().to_frame('Load')

# Create time-based aggregates
five = df_grouped['Load'].resample('5T').sum().to_frame('total_load')
quarter = df_grouped['Load'].resample('15T').sum().to_frame('total_load')
hourly = df_grouped['Load'].resample('H').sum().to_frame('total_load')
daily = df_grouped['Load'].resample('D').sum().to_frame('total_load')

# Reset index for plotting (add datetime column back)
quarter = quarter.reset_index()

print(f"✓ Created aggregates:")
print(f"  - 5-minute: {len(five):,} points")
print(f"  - 15-minute: {len(quarter):,} points")
print(f"  - Hourly: {len(hourly):,} points")
print(f"  - Daily: {len(daily):,} points")
print("\nVariables in memory: five, quarter, hourly, daily, df_grouped, df_all")


In [ ]:
#DELETE VARS FROM MEMORY 

del daily, df_all, five, hourly, MASTER_PATH


### Graph

In [ ]:
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd


#PLOTS

x = quarter.datetime
y = quarter.total_load


    # TODO: DAILY
# Filter for one day (first day in dataset)
one_day = quarter[quarter['datetime'].dt.date == quarter['datetime'].dt.date.iloc[0]]

# --- PLOT ---
plt.figure(figsize=(10, 5))
plt.plot(one_day['datetime'], one_day['total_load'],
         color='tab:blue', linewidth=2, marker='.', markersize=6)

# --- LABELS AND TITLE ---
plt.title(f'Total Load on {one_day["datetime"].dt.date.iloc[0]} (15 Minute Aggregation)', fontsize=14)
plt.xlabel('Time', fontsize=12)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))

plt.ylabel('Total Load', fontsize=12, rotation= 360, labelpad = 40)  # proper y-axis rotation

# --- ROTATE X-TICK LABELS ---
plt.xticks(rotation=45, ha='right')  # rotate labels and align right

# --- GRID ---
plt.grid(True, linestyle='--', alpha=0.6)

# Adjust spacing so nothing intersects
plt.tight_layout()

plt.show()

#     # TODO: MONTHLY 


jan = quarter[quarter['datetime'].dt.month == 1].copy()
plt.figure(figsize=(12, 4))
plt.plot(jan['datetime'], jan['total_load'], color='tab:orange', linewidth=1, alpha=0.8)
plt.title('January 2023 Total Load (15-minute Aggregation)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Load', fontsize=12)
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

#     # TODO: YEARLY 


plt.figure(figsize=(10, 5))
plt.plot(one_day['datetime'], one_day['total_load'],
         color='tab:blue', linewidth=2, marker='.', markersize=6)

# --- LABELS AND TITLE ---
plt.title('Total Load 2023 (15 min)', fontsize=14)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Total Load', fontsize=12, rotation= 360, labelpad = 40)  # proper y-axis rotation
plt.xticks(rotation=45, ha='right')  # rotate labels and align right
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


#### Spline

In [ ]:
from scipy.interpolate import UnivariateSpline


# Aggregate to daily means for a stable spline fit
daily = quarter.set_index('datetime')['total_load'].resample('D').mean().reset_index()

# Convert dates to numeric values for spline fitting
x_num = mdates.date2num(daily['datetime'])
y_daily = daily['total_load'].values

# Fit a smoothing spline (adjust `s` to change smoothness)
s = len(x_num) * np.var(y_daily) * 0.5
spline = UnivariateSpline(x_num, y_daily, s=s)

# Evaluate spline on a dense grid for a smooth curve
x_dense = np.linspace(x_num.min(), x_num.max(), 1000)
y_smooth = spline(x_dense)
x_dense_dates = mdates.num2date(x_dense)

# Plot original 15-min data (light), daily means (points), and spline (smooth)
plt.figure(figsize=(12, 4))
plt.plot(quarter['datetime'], quarter['total_load'], color='lightgray', linewidth=0.7, alpha=0.6, label='15-min raw')
plt.plot(daily['datetime'], daily['total_load'], 'o', color='tab:orange', markersize=4, label='Daily mean')
plt.plot(x_dense_dates, y_smooth, color='tab:blue', linewidth=2, label='Spline (daily fit)')
plt.title('Total Load 2023 with Spline')
plt.xlabel('Date')
plt.ylabel('Total Load')
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()







In [ ]:
# Multi-year load visualization (2020-2025) - Memory optimized

# Load data in chunks to reduce memory usage
df_multi = pd.read_parquet(project_root / "1_LIB" / "master" / "master.parquet")

# Ensure datetime column is properly formatted
if 'datetime' in df_multi.columns:
    df_multi['datetime'] = pd.to_datetime(df_multi['datetime'], utc=True)
elif 'Time Stamp' in df_multi.columns:
    df_multi['datetime'] = pd.to_datetime(df_multi['Time Stamp'], utc=True)

# Filter for years 2020-2025 early to reduce memory footprint
df_multi = df_multi[(df_multi['datetime'].dt.year >= 2020) & (df_multi['datetime'].dt.year <= 2025)].copy()
print(f"Filtered data: {len(df_multi)} rows from 2020-2025")

# Aggregate to 15-minute intervals more efficiently
df_multi = df_multi.set_index('datetime')

# Resample directly without redundant groupby - sum all loads per 15-min interval
quarter_multi = df_multi['Load'].resample('15T').sum().to_frame('total_load').reset_index()

# Add year column for grouping
quarter_multi['year'] = quarter_multi['datetime'].dt.year

print(f"Aggregated to {len(quarter_multi)} 15-minute intervals")

# Clear intermediate dataframe to free memory
del df_multi
import gc
gc.collect()

# Prepare spline fit for overall trend
daily_multi = quarter_multi.set_index('datetime')['total_load'].resample('D').mean().reset_index()

# Convert dates to numeric values for spline fitting
x_num = mdates.date2num(daily_multi['datetime'])
y_daily = daily_multi['total_load'].values

# Fit a smoothing spline (adjust `s` to change smoothness)
s = len(x_num) * np.var(y_daily) * 0.5
spline = UnivariateSpline(x_num, y_daily, s=s)

# Evaluate spline on a dense grid for a smooth curve
x_dense = np.linspace(x_num.min(), x_num.max(), 1000)
y_smooth = spline(x_dense)
x_dense_dates = mdates.num2date(x_dense)

# COVID timeline markers for New York
# Source: NY Times - "New York Coronavirus Map and Case Count" 
# First confirmed case: March 1, 2020 (https://www.nytimes.com/interactive/2021/us/new-york-covid-cases.html)
covid_start = pd.Timestamp('2020-03-01', tz='UTC')

# Source: Governor Cuomo's Executive Order - State disaster emergency ended June 25, 2021
# (https://www.ebglaw.com/insights/publications/the-state-of-emergency-in-new-york-is-over-what-that-means-for-employers)
covid_end = pd.Timestamp('2021-06-25', tz='UTC')

# Plot each year with different colors
plt.figure(figsize=(14, 6))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

for i, year in enumerate(range(2020, 2026)):
    year_data = quarter_multi[quarter_multi['year'] == year]
    if not year_data.empty:
        plt.plot(year_data['datetime'], year_data['total_load'], 
                color=colors[i], linewidth=1, alpha=0.5, label=str(year))
        print(f"  {year}: {len(year_data)} data points")

# Add daily means and spline
plt.plot(daily_multi['datetime'], daily_multi['total_load'], 'o', color='black', markersize=2, alpha=0.4, label='Daily mean')
plt.plot(x_dense_dates, y_smooth, color='darkred', linewidth=3, label='Spline (daily fit)', zorder=10)

# Add COVID markers
plt.axvline(covid_start, color='crimson', linestyle='--', linewidth=2, alpha=0.8, label='COVID Start (Mar 1, 2020)')
plt.axvline(covid_end, color='forestgreen', linestyle='--', linewidth=2, alpha=0.8, label='COVID End (Jun 25, 2021)')

plt.title('Total Load 2020-2025 with Spline (15-minute Aggregation)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Load', fontsize=12)
plt.legend(loc='best', fontsize=9)
plt.grid(True, linestyle='--', alpha=0.4)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


#### Significant Events Affecting Load

In [ ]:
# Multi-year load with significant events that affect electricity demand

# Create figure
plt.figure(figsize=(16, 7))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

# Plot yearly data
for i, year in enumerate(range(2020, 2026)):
    year_data = quarter_multi[quarter_multi['year'] == year]
    if not year_data.empty:
        plt.plot(year_data['datetime'], year_data['total_load'], 
                color=colors[i], linewidth=1, alpha=0.4, label=str(year))

# Add spline trend
plt.plot(x_dense_dates, y_smooth, color='darkred', linewidth=2.5, label='Spline trend', zorder=10)

# === SIGNIFICANT EVENTS WITH VERIFIED SOURCES ===

# 1. COVID-19 Pandemic Start
# Source: NY Times COVID Tracking & NY State Department of Health
# First confirmed case in New York: March 1, 2020
# https://www.nytimes.com/interactive/2021/us/new-york-covid-cases.html
covid_start = pd.Timestamp('2020-03-01', tz='UTC')
plt.axvline(covid_start, color='crimson', linestyle='--', linewidth=2, alpha=0.7, 
            label='COVID-19 Pandemic Start (Mar 1, 2020)')

# 2. COVID-19 Emergency Declaration Ended
# Source: Governor Cuomo Executive Order No. 210.29
# State disaster emergency ended: June 25, 2021
# https://www.ebglaw.com/insights/publications/the-state-of-emergency-in-new-york-is-over-what-that-means-for-employers
covid_end = pd.Timestamp('2021-06-25', tz='UTC')
plt.axvline(covid_end, color='forestgreen', linestyle='--', linewidth=2, alpha=0.7,
            label='COVID Emergency Ended (Jun 25, 2021)')

# 3. Summer 2022 Heat Wave
# Extreme heat event affecting NYC and surrounding areas: July 18-23, 2022
#https://www.nyserda.ny.gov/Featured-Stories/Protecting-New-Yorkers-from-Extreme-Heat
heat_wave_2022 = pd.Timestamp('2022-07-20', tz='UTC')
plt.axvline(heat_wave_2022, color='darkorange', linestyle='-.', linewidth=2, alpha=0.7,
            label='Summer 2022 Heat Wave (Jul 20, 2022)')

# 4. Winter Storm Elliott
# Major winter storm affecting Northeast: December 23, 2022
# https://www.ferc.gov/news-events/news/ferc-nerc-release-final-report-lessons-winter-storm-elliott
winter_storm_2022 = pd.Timestamp('2022-12-23', tz='UTC')
plt.axvline(winter_storm_2022, color='steelblue', linestyle='-.', linewidth=2, alpha=0.7,
            label='Winter Storm Elliott (Dec 23, 2022)')

# 5. Summer 2023 Canadian Wildfire Smoke
# Wildfire smoke blanketed NYC, reduced cooling loads: June 7-8, 2023
# https://www.rutgers.edu/news/canadian-wildfire-smoke-cooled-new-york-3-degrees-and-trapped-air-toxicants
wildfire_smoke = pd.Timestamp('2023-06-07', tz='UTC')
plt.axvline(wildfire_smoke, color='dimgray', linestyle='-.', linewidth=2, alpha=0.7,
            label='Canadian Wildfire Smoke (Jun 7, 2023)')

# 6. July 2023 Heat Wave
# Record-breaking temperatures in NYC area: July 28, 2023
# https://www.health.ny.gov/press/releases/2023/2023-07-25_heat_advisory.htm
heat_wave_2023 = pd.Timestamp('2023-07-28', tz='UTC')
plt.axvline(heat_wave_2023, color='red', linestyle='-.', linewidth=2, alpha=0.7,
            label='July 2023 Heat Wave (Jul 28, 2023)')

# 7. January 2024 Cold Snap
# Extreme cold affecting Northeast: January 13-16, 2024
# https://science.nasa.gov/earth/earth-observatory/arctic-chill-sweeps-us-152333/
cold_snap_2024 = pd.Timestamp('2024-01-15', tz='UTC')
plt.axvline(cold_snap_2024, color='navy', linestyle='-.', linewidth=2, alpha=0.7,
            label='January 2024 Cold Snap (Jan 15, 2024)')

# Formatting
plt.title('New York Total Load 2020-2025: Significant Events Affecting Electricity Demand', fontsize=15, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Load', fontsize=12)
plt.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.3)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nEvent Summary:")
print("=" * 80)
print("1. COVID-19 Pandemic Start (Mar 1, 2020) - Reduced commercial/industrial loads")
print("2. COVID Emergency Ended (Jun 25, 2021) - Return to normal operations")
print("3. Summer 2022 Heat Wave (Jul 20, 2022) - Increased cooling demand")
print("4. Winter Storm Elliott (Dec 23, 2022) - Increased heating demand")
print("5. Canadian Wildfire Smoke (Jun 7, 2023) - Reduced visibility, altered loads")
print("6. July 2023 Heat Wave (Jul 28, 2023) - Record cooling demand")
print("7. January 2024 Cold Snap (Jan 15, 2024) - Extreme heating demand")
print("=" * 80)


#### Create Animation - Multi-Year Load Evolution

In [ ]:
# Create animated visualization of multi-year load evolution
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# Prepare data - use daily aggregates for smoother animation
daily_anim = quarter_multi.set_index('datetime')['total_load'].resample('D').mean().reset_index()
daily_anim['year'] = daily_anim['datetime'].dt.year
daily_anim['day_of_year'] = daily_anim['datetime'].dt.dayofyear

print(f"Creating animation with {len(daily_anim)} daily data points...")

# Prepare spline for animation (using the same spline from earlier)
# Convert x_dense_dates to pandas DatetimeIndex for easier comparison
print("Preparing spline data for animation...")
x_dense_dates_series = pd.DatetimeIndex(x_dense_dates)

# Create figure for animation
fig, ax = plt.subplots(figsize=(14, 6))

def animate(frame):
    ax.clear()
    
    # Calculate which date we're showing (show data progressively through timeline)
    # Show data up to current frame - slower progression (5 days at a time instead of 30)
    current_data = daily_anim.iloc[:frame+5]
    
    if len(current_data) == 0:
        return
    
    # Plot each year's data up to current point with distinct colors
    colors_map = {
        2020: 'tab:blue', 
        2021: 'tab:orange', 
        2022: 'tab:green', 
        2023: 'tab:red', 
        2024: 'tab:purple', 
        2025: 'tab:brown'
    }
    
    for year in range(2020, 2026):
        year_data = current_data[current_data['year'] == year]
        if not year_data.empty:
            ax.plot(year_data['datetime'], year_data['total_load'], 
                   color=colors_map.get(year, 'gray'), linewidth=2, 
                   alpha=0.7, label=f'{year}')
    
    # Add spline trend line up to current date
    current_date = current_data['datetime'].iloc[-1]
    spline_mask = x_dense_dates_series <= current_date
    if spline_mask.any():
        ax.plot(x_dense_dates_series[spline_mask], y_smooth[spline_mask], 
               color='darkred', linewidth=3, label='Spline Trend', zorder=10, alpha=0.9)
    
    # Add event markers with labels if we've reached them
    y_max = daily_anim['total_load'].max() * 1.03
    y_min = 3000  # Set minimum Y-axis to 2000 as requested
    
    # COVID start - bold grey
    covid_start_date = pd.Timestamp('2020-03-01', tz='UTC')
    if current_date >= covid_start_date:
        ax.axvline(covid_start_date, 
                  color='#404040', linestyle='--', linewidth=3, alpha=0.8, zorder=5)
        # Add label at top of plot
        ax.text(covid_start_date, y_max, 'COVID-19\nPandemic Start\n(Mar 1, 2020)', 
               ha='center', va='top', fontsize=9, color='#404040', 
               bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#404040', alpha=0.9),
               zorder=6, weight='bold')
    
    # COVID end - bold grey
    covid_end_date = pd.Timestamp('2021-06-25', tz='UTC')
    if current_date >= covid_end_date:
        ax.axvline(covid_end_date, 
                  color='#404040', linestyle='--', linewidth=3, alpha=0.8, zorder=5)
        # Add label at top of plot
        ax.text(covid_end_date, y_max, 'COVID Emergency\nEnded\n(Jun 25, 2021)', 
               ha='center', va='top', fontsize=9, color='#404040',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#404040', alpha=0.9),
               zorder=6, weight='bold')
    
    # Formatting
    ax.set_title(f'NY Total Load Evolution with Spline: 2020-2025\nShowing data through {current_date.strftime("%B %d, %Y")}', 
                fontsize=14, pad=15, weight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Daily Avg Total Load (MW)', fontsize=12)
    
    # Move legend to top right with color-coded years
    ax.legend(loc='upper right', fontsize=9, framealpha=0.95, 
             title='Year / Trend', title_fontsize=9)
    
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45, ha='right')
    
    # Set consistent axis limits - Y-axis starts at 2000
    ax.set_xlim(daily_anim['datetime'].min(), daily_anim['datetime'].max())
    ax.set_ylim(y_min, y_max)

# Create animation - sample every 5 days for slower, smoother progression
frames = range(0, len(daily_anim), 5)
print(f"Generating {len(frames)} frames (slower progression for smoother animation)...")

anim = FuncAnimation(fig, animate, frames=frames, interval=150, repeat=True)

# Save animation
output_path = project_root / "2_FIGURES" / "FIGURES" / "multi_year_load_evolution.gif"
print(f"Saving animation to: {output_path}")

writer = PillowWriter(fps=8)  # Reduced from 10 fps to 8 fps for slower playback
anim.save(output_path, writer=writer)

plt.close(fig)

print(f"✓ Animation saved successfully!")
print(f"  File: {output_path}")
print(f"  Frames: {len(frames)}")
print(f"  Features:")
print(f"    - Spline trend (dark red)")
print(f"    - Bold grey event markers (COVID events)")
print(f"    - Color-coded years in legend")
print(f"    - Y-axis starting at 2000 MW")
print(f"    - Legend positioned top right")
print(f"  Playback: Slower progression (8 fps, 5-day intervals)")

# Display the animation
display(Image(filename=str(output_path)))


### Model Comparison: Linear Regression, XGBoost, and SVR (2023)

In [ ]:
# Create Summary Statistics for Hourly, Month, Year 

periods = {'Jan 1': one_day, 'January': jan, 'Year': quarter}
rows = []
for name, df in periods.items():
    if df.empty:
        rows.append({'period': name, 'count': 0})
        continue
    s = df['total_load'].describe()
    peak_idx = df['total_load'].idxmax()
    peak_time = df.loc[peak_idx, 'datetime']
    peak_val = df.loc[peak_idx, 'total_load']
    rows.append({
        'period': name,
        'count': int(s['count']),
        'mean': round(s['mean'], 2),
        'std': round(s['std'], 2),
        'min': round(s['min'], 2),
        '25%': round(s['25%'], 2),
        '50%': round(s['50%'], 2),
        '75%': round(s['75%'], 2),
        'max': round(s['max'], 2),
        'sum': round(df['total_load'].sum(), 2),
        'peak_time': pd.to_datetime(peak_time).strftime('%Y-%m-%d %H:%M:%S'),
        'peak_value': round(peak_val, 2),
    })

summary_stats = pd.DataFrame(rows).set_index('period')
summary_stats


In [ ]:
# Load master parquet file with merged NYISO load and weather data
from pathlib import Path
project_root = Path(__file__).parent.parent.parent if '__file__' in dir() else Path.cwd().parent.parent

MASTER_PATH = project_root / "1_LIB" / "master" / "master.parquet"
print(f"Loading data from: {MASTER_PATH}")

# Load the master parquet file
df = pd.read_parquet(MASTER_PATH)

# Standardize datetime column name
if 'datetime' in df.columns:
    df['Time'] = pd.to_datetime(df['datetime'], utc=True)
elif 'Time Stamp' in df.columns:
    df['Time'] = pd.to_datetime(df['Time Stamp'], utc=True)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Load full dataset and create train/val/test splits
import pandas as pd
import numpy as np
from pathlib import Path
import time
import gc

print("="*80)
print("LOADING FULL DATASET FOR TRAIN/VAL/TEST SPLIT")
print("="*80)

# Configuration for splits
split_config = {
    "train_start": 2001,
    "train_end": 2021,
    "val_year": 2022,
    "test_years": [2023, 2024, 2025]
}

print(f"\nSplit Configuration:")
print(f"  Training:   {split_config['train_start']}-{split_config['train_end']} ({split_config['train_end'] - split_config['train_start'] + 1} years)")
print(f"  Validation: {split_config['val_year']} (1 year)")
print(f"  Test:       {split_config['test_years']} ({len(split_config['test_years'])} years)")

# Load full master parquet
MASTER_PATH = project_root / "1_LIB" / "master" / "master.parquet"
print(f"\n[1/4] Loading full dataset from: {MASTER_PATH.name}")
start = time.time()
df_full = pd.read_parquet(MASTER_PATH, columns=['datetime', 'Load'])
print(f"  ✓ Loaded {len(df_full):,} rows in {time.time()-start:.1f}s")

# Convert datetime
print("\n[2/4] Processing datetime...")
df_full['datetime'] = pd.to_datetime(df_full['datetime'], utc=True)
df_full = df_full.set_index('datetime')

# Aggregate to 15-minute intervals (sum across all zones)
print("\n[3/4] Aggregating to 15-minute intervals...")
start = time.time()
df_15min = df_full['Load'].resample('15T').sum().to_frame('total_load').reset_index()
print(f"  ✓ Aggregated to {len(df_15min):,} rows in {time.time()-start:.1f}s")

# Add year column for splitting
df_15min['year'] = df_15min['datetime'].dt.year

# Create splits
print("\n[4/4] Creating train/val/test splits...")
train_data = df_15min[
    (df_15min['year'] >= split_config['train_start']) & 
    (df_15min['year'] <= split_config['train_end'])
].copy()

val_data = df_15min[df_15min['year'] == split_config['val_year']].copy()

test_data = df_15min[df_15min['year'].isin(split_config['test_years'])].copy()

# Print summary
print("\n" + "="*80)
print("DATASET SPLIT SUMMARY")
print("="*80)
print(f"{'Split':<12} {'Years':<20} {'Rows':<12} {'% of Total':<12}")
print("-"*80)
total_rows = len(df_15min)
print(f"{'Train':<12} {split_config['train_start']}-{split_config['train_end']:<16} {len(train_data):>10,}  {len(train_data)/total_rows*100:>10.1f}%")
print(f"{'Validation':<12} {split_config['val_year']:<20} {len(val_data):>10,}  {len(val_data)/total_rows*100:>10.1f}%")
print(f"{'Test':<12} {str(split_config['test_years']):<20} {len(test_data):>10,}  {len(test_data)/total_rows*100:>10.1f}%")
print("-"*80)
print(f"{'TOTAL':<12} {'2001-2025':<20} {total_rows:>10,}  {100.0:>10.1f}%")
print("="*80)

# Memory cleanup
del df_full, df_15min
gc.collect()

print("\n✓ Variables created: train_data, val_data, test_data")
print("="*80 + "\n")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
import numpy as np

print("="*80)
print("TRAINING MODELS ON MULTI-YEAR DATA (2001-2025)")
print("="*80)

# ============================================================================
# MODEL 1: LINEAR REGRESSION
# ============================================================================
print("\n[1/3] Training Linear Regression...")
print("  Features: hour, day_of_week, month, day_of_year")
print("  Applied to: 15-minute intervals")

train_lr = train_data.set_index('datetime').copy()
val_lr = val_data.set_index('datetime').copy()
test_lr = test_data.set_index('datetime').copy()

# Feature engineering - time features
for df in [train_lr, val_lr, test_lr]:
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day_of_year'] = df.index.dayofyear

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(train_lr[['hour', 'day_of_week', 'month', 'day_of_year']], train_lr['total_load'])

# Predict on all splits
lr_train_pred = lr_model.predict(train_lr[['hour', 'day_of_week', 'month', 'day_of_year']])
lr_val_pred = lr_model.predict(val_lr[['hour', 'day_of_week', 'month', 'day_of_year']])
lr_test_pred = lr_model.predict(test_lr[['hour', 'day_of_week', 'month', 'day_of_year']])

# Calculate metrics
lr_train_mape = mean_absolute_percentage_error(train_lr['total_load'], lr_train_pred) * 100
lr_val_mape = mean_absolute_percentage_error(val_lr['total_load'], lr_val_pred) * 100
lr_test_mape = mean_absolute_percentage_error(test_lr['total_load'], lr_test_pred) * 100

print(f"  ✓ Train MAPE: {lr_train_mape:.2f}%")
print(f"  ✓ Val MAPE:   {lr_val_mape:.2f}%")
print(f"  ✓ Test MAPE:  {lr_test_mape:.2f}%")

# ============================================================================
# MODEL 2: XGBOOST
# ============================================================================
print("\n[2/3] Training XGBoost...")
print("  Features: hour, day_of_week, month + lag features (1,2,4,24,96)")
print("  Applied to: 15-minute intervals")

train_xgb = train_data.copy().set_index('datetime')
val_xgb = val_data.copy().set_index('datetime')
test_xgb = test_data.copy().set_index('datetime')

# Create lag features (15-min intervals)
# lag_1 = 15 minutes ago, lag_4 = 1 hour ago, lag_24 = 6 hours ago, lag_96 = 24 hours ago
for lag in [1, 2, 4, 24, 96]:
    train_xgb[f'lag_{lag}'] = train_xgb['total_load'].shift(lag)
    val_xgb[f'lag_{lag}'] = val_xgb['total_load'].shift(lag)
    test_xgb[f'lag_{lag}'] = test_xgb['total_load'].shift(lag)

# Add time features
for df in [train_xgb, val_xgb, test_xgb]:
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month

# Drop rows with NaN (from lag features)
train_xgb = train_xgb.dropna()
val_xgb = val_xgb.dropna()
test_xgb = test_xgb.dropna()

# Define feature columns
feature_cols_xgb = ['hour', 'day_of_week', 'month', 'lag_1', 'lag_2', 'lag_4', 'lag_24', 'lag_96']

# Train XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1
)
xgb_model.fit(train_xgb[feature_cols_xgb], train_xgb['total_load'], verbose=False)

# Predict on all splits
xgb_train_pred = xgb_model.predict(train_xgb[feature_cols_xgb])
xgb_val_pred = xgb_model.predict(val_xgb[feature_cols_xgb])
xgb_test_pred = xgb_model.predict(test_xgb[feature_cols_xgb])

# Calculate metrics
xgb_train_mape = mean_absolute_percentage_error(train_xgb['total_load'], xgb_train_pred) * 100
xgb_val_mape = mean_absolute_percentage_error(val_xgb['total_load'], xgb_val_pred) * 100
xgb_test_mape = mean_absolute_percentage_error(test_xgb['total_load'], xgb_test_pred) * 100

print(f"  ✓ Train MAPE: {xgb_train_mape:.2f}%")
print(f"  ✓ Val MAPE:   {xgb_val_mape:.2f}%")
print(f"  ✓ Test MAPE:  {xgb_test_mape:.2f}%")

# ============================================================================
# MODEL 3: SVR (Support Vector Regression)
# ============================================================================
print("\n[3/3] Training SVR...")
print("  Features: day_of_year, day_of_week, month (scaled)")
print("  Applied to: Daily aggregates")

# Aggregate to daily first (SVR works better on daily data)
train_daily_svr = train_data.set_index('datetime')['total_load'].resample('D').mean().reset_index()
val_daily_svr = val_data.set_index('datetime')['total_load'].resample('D').mean().reset_index()
test_daily_svr = test_data.set_index('datetime')['total_load'].resample('D').mean().reset_index()

# Feature engineering - seasonal features
for df in [train_daily_svr, val_daily_svr, test_daily_svr]:
    df['day_of_year'] = df['datetime'].dt.dayofyear
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

# Scale features (SVR requires scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_daily_svr[['day_of_year', 'day_of_week', 'month']])
X_val_scaled = scaler.transform(val_daily_svr[['day_of_year', 'day_of_week', 'month']])
X_test_scaled = scaler.transform(test_daily_svr[['day_of_year', 'day_of_week', 'month']])

# Train SVR
svr_model = SVR(kernel='rbf', C=100, gamma=0.001)
svr_model.fit(X_train_scaled, train_daily_svr['total_load'])

# Predict (daily predictions)
svr_train_pred = svr_model.predict(X_train_scaled)
svr_val_pred = svr_model.predict(X_val_scaled)
svr_test_pred = svr_model.predict(X_test_scaled)

# Calculate metrics
svr_train_mape = mean_absolute_percentage_error(train_daily_svr['total_load'], svr_train_pred) * 100
svr_val_mape = mean_absolute_percentage_error(val_daily_svr['total_load'], svr_val_pred) * 100
svr_test_mape = mean_absolute_percentage_error(test_daily_svr['total_load'], svr_test_pred) * 100

print(f"  ✓ Train MAPE: {svr_train_mape:.2f}%")
print(f"  ✓ Val MAPE:   {svr_val_mape:.2f}%")
print(f"  ✓ Test MAPE:  {svr_test_mape:.2f}%")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY (MAPE %)")
print("="*80)
print(f"{'Model':<20} {'Train':<12} {'Val':<12} {'Test':<12}")
print("-"*80)
print(f"{'Linear Regression':<20} {lr_train_mape:>10.2f}%  {lr_val_mape:>10.2f}%  {lr_test_mape:>10.2f}%")
print(f"{'XGBoost':<20} {xgb_train_mape:>10.2f}%  {xgb_val_mape:>10.2f}%  {xgb_test_mape:>10.2f}%")
print(f"{'SVR (Daily)':<20} {svr_train_mape:>10.2f}%  {svr_val_mape:>10.2f}%  {svr_test_mape:>10.2f}%")
print("="*80)

# Determine best model based on validation performance
best_model_name = min(
    [('Linear Regression', lr_val_mape), 
     ('XGBoost', xgb_val_mape), 
     ('SVR', svr_val_mape)],
    key=lambda x: x[1]
)[0]
print(f"\n🏆 Best model (by validation MAPE): {best_model_name}")
print("="*80 + "\n")

In [ ]:
# Multi-Year Model Comparison Animation: 2020-2025 with Events and Spline
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

print("="*80)
print("MULTI-YEAR MODEL COMPARISON: 2020-2025 with Events & Spline")
print("="*80)

# Define MAPE calculation function
def calc_mape(actual, pred):
    mask = actual != 0
    return (np.abs((actual[mask] - pred[mask]) / actual[mask])).mean() * 100

# Prepare multi-year daily data
print("\n[1/3] Preparing data...")
daily_all_years = quarter_multi.set_index('datetime')['total_load'].resample('D').mean().reset_index()
daily_all_years.columns = ['datetime', 'actual_load']
daily_all_years['year'] = daily_all_years['datetime'].dt.year

# Train models per year and generate predictions
print("\n[2/3] Training models...")
all_predictions = []

for test_year in range(2020, 2026):
    print(f"  Processing {test_year}...")
    train_years = list(range(2020, test_year)) if test_year > 2020 else [2020]
    
    if test_year == 2020:
        train_data = quarter_multi[quarter_multi['year'] == 2020]
        test_data = quarter_multi[quarter_multi['year'] == 2020]
    else:
        train_data = quarter_multi[quarter_multi['year'].isin(train_years)]
        test_data = quarter_multi[quarter_multi['year'] == test_year]
    
    # Linear Regression
    train_lr = train_data.set_index('datetime').copy()
    test_lr = test_data.set_index('datetime').copy()
    for df in [train_lr, test_lr]:
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['month'] = df.index.month
        df['day_of_year'] = df.index.dayofyear
    
    lr_temp = LinearRegression()
    lr_temp.fit(train_lr[['hour', 'day_of_week', 'month', 'day_of_year']], train_lr['total_load'])
    lr_pred = lr_temp.predict(test_lr[['hour', 'day_of_week', 'month', 'day_of_year']])
    
    # XGBoost
    train_xgb = train_data.copy().set_index('datetime')
    test_xgb = test_data.copy().set_index('datetime')
    for lag in [1, 2, 4, 24, 96]:
        train_xgb[f'lag_{lag}'] = train_xgb['total_load'].shift(lag)
        test_xgb[f'lag_{lag}'] = test_xgb['total_load'].shift(lag)
    for df in [train_xgb, test_xgb]:
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['month'] = df.index.month
    
    train_xgb = train_xgb.dropna()
    test_xgb = test_xgb.dropna()
    feature_cols_xgb = ['hour', 'day_of_week', 'month', 'lag_1', 'lag_2', 'lag_4', 'lag_24', 'lag_96']
    
    xgb_temp = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
    xgb_temp.fit(train_xgb[feature_cols_xgb], train_xgb['total_load'], verbose=False)
    xgb_pred = xgb_temp.predict(test_xgb[feature_cols_xgb])
    
    # SVR
    from sklearn.svm import SVR
    from sklearn.preprocessing import StandardScaler
    
    train_daily_svr = train_data.set_index('datetime')['total_load'].resample('D').mean().reset_index()
    test_daily_svr = test_data.set_index('datetime')['total_load'].resample('D').mean().reset_index()
    for df in [train_daily_svr, test_daily_svr]:
        df['day_of_year'] = df['datetime'].dt.dayofyear
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['month'] = df['datetime'].dt.month
    
    scaler_temp = StandardScaler()
    X_train_scaled = scaler_temp.fit_transform(train_daily_svr[['day_of_year', 'day_of_week', 'month']])
    X_test_scaled = scaler_temp.transform(test_daily_svr[['day_of_year', 'day_of_week', 'month']])
    
    svr_temp = SVR(kernel='rbf', C=100, gamma=0.001)
    svr_temp.fit(X_train_scaled, train_daily_svr['total_load'])
    svr_pred_daily = svr_temp.predict(X_test_scaled)
    
    # Aggregate to daily
    lr_daily_temp = pd.DataFrame({'datetime': test_lr.index, 'lr_pred': lr_pred}).set_index('datetime')['lr_pred'].resample('D').mean().reset_index()
    xgb_daily_temp = pd.DataFrame({'datetime': test_xgb.index, 'xgb_pred': xgb_pred}).set_index('datetime')['xgb_pred'].resample('D').mean().reset_index()
    svr_daily_temp = pd.DataFrame({'datetime': test_daily_svr['datetime'], 'svr_pred': svr_pred_daily})
    
    year_preds = test_daily_svr[['datetime']].copy()
    year_preds = year_preds.merge(lr_daily_temp, on='datetime', how='left')
    year_preds = year_preds.merge(xgb_daily_temp, on='datetime', how='left')
    year_preds = year_preds.merge(svr_daily_temp, on='datetime', how='left')
    all_predictions.append(year_preds)

predictions_all = pd.concat(all_predictions, ignore_index=True).ffill().bfill()
comparison_multi = daily_all_years.merge(predictions_all[['datetime', 'lr_pred', 'xgb_pred', 'svr_pred']], on='datetime', how='left').ffill().bfill()

lr_mape_all = calc_mape(comparison_multi['actual_load'].values, comparison_multi['lr_pred'].values)
xgb_mape_all = calc_mape(comparison_multi['actual_load'].values, comparison_multi['xgb_pred'].values)
svr_mape_all = calc_mape(comparison_multi['actual_load'].values, comparison_multi['svr_pred'].values)

print(f"\n{'='*80}")
print(f"OVERALL 2020-2025 PERFORMANCE (MAPE):")
print(f"  Linear Regression: {lr_mape_all:.2f}%")
print(f"  XGBoost:          {xgb_mape_all:.2f}%")
print(f"  SVR:              {svr_mape_all:.2f}%")
print(f"{'='*80}")


In [ ]:
# Create Animation
print(f"\n[3/3] Creating animation...")
fig, ax = plt.subplots(figsize=(18, 9))
rolling_window = 30

# Convert x_dense_dates to pandas DatetimeIndex
x_dense_dates_series = pd.DatetimeIndex(x_dense_dates)

# Define events with staggered heights to prevent overlap
events = [
    {'date': covid_start, 'label': 'COVID-19\nStart', 'color': 'crimson', 'height': 0.98},
    {'date': covid_end, 'label': 'COVID\nEnd', 'color': 'forestgreen', 'height': 0.91},
    {'date': heat_wave_2022, 'label': '2022\nHeat Wave', 'color': 'darkorange', 'height': 0.84},
    {'date': winter_storm_2022, 'label': 'Winter\nStorm', 'color': 'steelblue', 'height': 0.98},
    {'date': wildfire_smoke, 'label': 'Wildfire\nSmoke', 'color': 'dimgray', 'height': 0.91},
    {'date': heat_wave_2023, 'label': '2023\nHeat Wave', 'color': 'red', 'height': 0.84},
    {'date': cold_snap_2024, 'label': '2024\nCold Snap', 'color': 'navy', 'height': 0.98}
]

def animate_multi(frame):
    ax.clear()
    end_idx = min(frame * 3 + rolling_window, len(comparison_multi))
    current_data = comparison_multi.iloc[:end_idx]
    
    if len(current_data) < 2:
        return
    
    current_date = current_data['datetime'].iloc[-1]
    y_max = comparison_multi['actual_load'].max() * 1.05
    y_min = 3000
    
    # STEP 1: Plot background model lines first
    ax.plot(current_data['datetime'], current_data['lr_pred'], 
            color='tab:red', linewidth=2.5, label='Linear Regression', 
            alpha=0.7, linestyle='--')
    ax.plot(current_data['datetime'], current_data['svr_pred'], 
            color='tab:blue', linewidth=2.5, label='SVR', 
            alpha=0.7, linestyle='--')
    
    # STEP 2: Plot XGBoost (middle layer)
    ax.plot(current_data['datetime'], current_data['xgb_pred'], 
            color='#006400', linewidth=2.5, label='XGBoost', 
            alpha=0.9, linestyle='-')
    
    # STEP 3: Plot Actual Load AFTER XGBoost so it appears on top
    ax.plot(current_data['datetime'], current_data['actual_load'], 
            color='lime', linewidth=2.5, label='Actual Load', 
            alpha=0.8)
    
    # STEP 4: Plot event vertical lines (before text boxes)
    for event in events:
        if current_date >= event['date']:
            ax.axvline(event['date'], color=event['color'], 
                      linestyle='-.', linewidth=2.5, alpha=0.6)
    
    # STEP 5: Plot event text boxes LAST so they overlay everything
    for event in events:
        if current_date >= event['date']:
            ax.text(event['date'], y_max * event['height'], event['label'], 
                   ha='center', va='top', fontsize=11, color=event['color'],
                   bbox=dict(boxstyle='round,pad=0.6', facecolor='white',
                            edgecolor=event['color'], alpha=0.95, linewidth=2), 
                   weight='bold')
    
    # STEP 6: MAPE text box - plot LAST
    if len(current_data) >= rolling_window:
        recent_data = current_data.tail(rolling_window)
        lr_rolling = calc_mape(recent_data['actual_load'].values, recent_data['lr_pred'].values)
        xgb_rolling = calc_mape(recent_data['actual_load'].values, recent_data['xgb_pred'].values)
        svr_rolling = calc_mape(recent_data['actual_load'].values, recent_data['svr_pred'].values)
        
        textstr = f'Rolling MAPE ({rolling_window}-day):\nLR:      {lr_rolling:6.2f}%\nXGBoost: {xgb_rolling:6.2f}%\nSVR:     {svr_rolling:6.2f}%'
        props = dict(boxstyle='round,pad=0.8', facecolor='wheat', alpha=0.95,
                    edgecolor='black', linewidth=2)
        ax.text(0.98, 0.02, textstr, transform=ax.transAxes, fontsize=13,
                verticalalignment='bottom', horizontalalignment='right', 
                bbox=props, family='monospace', weight='bold')
    
    # Formatting
    ax.set_title(f'Multi-Year Model Comparison: 2020-2025\nShowing data through {current_date.strftime("%B %d, %Y")}',
                fontsize=16, pad=15, weight='bold')
    ax.set_xlabel('Date', fontsize=13)
    ax.set_ylabel('Daily Avg Total Load (MW)', fontsize=13)
    
    # Legend
    ax.legend(loc='upper right', fontsize=11, framealpha=0.95, 
             ncol=1, edgecolor='black', fancybox=True, shadow=True)
    
    ax.grid(True, linestyle='--', alpha=0.25, linewidth=0.8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(fontsize=11)
    ax.set_xlim(comparison_multi['datetime'].min(), comparison_multi['datetime'].max())
    ax.set_ylim(y_min, y_max)
    plt.tight_layout()

frames_multi = range(0, len(comparison_multi) // 3)
anim_multi = FuncAnimation(fig, animate_multi, frames=frames_multi, interval=100, repeat=True)

output_path_multi = project_root / "2_FIGURES" / "FIGURES" / "model_comparison_2020_2025_with_events.gif"
writer_multi = PillowWriter(fps=10)
anim_multi.save(output_path_multi, writer=writer_multi)
plt.close(fig)

print(f"\n✓ Animation saved: {output_path_multi.name}")
print(f"  Frames: {len(frames_multi)}, Features: 2020-2025, Events (7), Rolling MAPE")
print(f"{'='*80}\n")

display(Image(filename=str(output_path_multi)))